# Model Training

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1):
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed:
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Disable all auto-JIT clustering at the process level:
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

# Enable deterministic operations for reproducibility:
# os.environ["TF_DETERMINISTIC_OPS"] = "1"
# os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# If it fails to determine best cudnn convolution algorithm:
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Suppress TensorFlow logging (1: INFO, 2: WARNING, 3: ERROR):
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized imports for the project

## 2. Run Parameters 

In [ ]:
EPOCHS = 5
BATCH_SIZE = 64

DATA_SEED = 0
TRAIN_SEED = 0

# Set Python, NumPy, Keras and TensorFlow seeds
set_random_seed(TRAIN_SEED)

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
tf.config.experimental.enable_op_determinism()

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
USE_JIT_COMPILE = False


In [ ]:
POLICY = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/train_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels(s008_path="./data/s008", s009_path="./data/s009")

(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_s008_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)


## 4. Model Definition

In [ ]:
def build_model(show_summary: bool = True) -> Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    #! Lambda has deserialization issues, so providing the output shape is necessary

    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    x = layers.Conv1D(
        filters=128,
        kernel_size=9,
        padding="same",
        data_format="channels_last",
        activation=None,
        use_bias=False,
        kernel_initializer=initializer,
        name="conv1d_0",
    )(combined)
    x = layers.BatchNormalization(name="conv1d_0_bn")(x)
    x = layers.Activation("silu", name="conv1d_0_act")(x)
    x = layers.MaxPooling1D(pool_size=4, name="max_pool_0")(x)

    x = layers.Conv1D(
        filters=256,
        kernel_size=4,
        padding="same",
        data_format="channels_last",
        activation=None,
        use_bias=False,
        kernel_initializer=initializer,
        name="conv1d_1",
    )(x)
    x = layers.BatchNormalization(name="conv1d_1_bn")(x)
    x = layers.Activation("tanh", name="conv1d_1_act")(x)
    x = layers.MaxPooling1D(pool_size=2, name="max_pool_1")(x)

    x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    x = layers.Dense(
        units=175,
        activation=None,
        kernel_initializer=initializer,
        name="dense_0",
    )(x)
    x = layers.Activation("tanh", name="dense_0_act")(x)
    x = layers.Dropout(rate=0.1, name="dense_0_dropout")(x)

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    optimizer = optimizers.AdamW(
        learning_rate=0.0028523343462769487,
        weight_decay=1e-4,
    )

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=USE_JIT_COMPILE,
    )

    return model


## Main

In [ ]:
try:
    # ——————————————————————————————————— Setup —————————————————————————————————— #
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        scaler_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR, study_name="model_training")

    # —————————————————————————————— Data Preprocessing ————————————————————————————— #
    coord_scaler = StandardScaler()

    coord_scaler.fit(x_s008_coord_train)
    x_s008_coord_train = coord_scaler.transform(x_s008_coord_train)
    x_s008_coord_val = coord_scaler.transform(x_s008_coord_val)
    s009_coord_input = coord_scaler.transform(s009_coord_input)
    s008_coord_input = coord_scaler.transform(s008_coord_input)

    scaler_path = os.path.join(scaler_dir, "coord_scaler.pkl")
    with open(scaler_path, "wb") as scaler_file:
        pickle.dump(coord_scaler, scaler_file)

    # —————————————————————————————— Train the Model ————————————————————————————— #

    model = build_model(show_summary=True)

    history = model.fit(
        x=[x_s008_lidar_train, x_s008_coord_train],
        y=y_s008_train,
        validation_data=([x_s008_lidar_val, x_s008_coord_val], y_s008_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks_model(
            backup_dir=os.path.join(backup_dir, "training"),
            checkpoint_dir=os.path.join(backup_dir, "checkpoints"),
            early_stopping_patience=10,
            reduce_lr_patience=3,
            #! Can cause high memory usage
            # tensorboard_logs=tensorboard_dir,
        ),
        verbose=2,
    )

    model.save(os.path.join(model_dir, "model.keras"))

    # ———————————————————————————————— Evaluations ———————————————————————————————— #
    s009_loss, s009_acc = model.evaluate(
        [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=2
    )

    s008_loss, s008_acc = model.evaluate(
        [s008_lidar_input, s008_coord_input], s008_y_train, batch_size=BATCH_SIZE, verbose=2
    )

    # ———————————————————————————————— Best Metrics ———————————————————————————————— #
    best_idx = int(np.argmax(history.history["val_accuracy"]))

    best_train_loss = float(history.history["loss"][best_idx])
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_train_acc = float(history.history["accuracy"][best_idx])
    best_val_acc = float(history.history["val_accuracy"][best_idx])

    # ——————————————————————————————— Save history ——————————————————————————————— #
    history_path = os.path.join(history_dir, "history.csv")
    history_data = {
        "epoch": list(range(1, len(history.history["loss"]) + 1)),
        "train_loss": history.history["loss"],
        "val_loss": history.history["val_loss"],
        "train_accuracy": history.history["accuracy"],
        "val_accuracy": history.history["val_accuracy"],
    }

    history_df = pd.DataFrame(history_data)
    history_df.to_csv(history_path, index=False)

    # ———————————————————————————————— Model Stats ——————————————————————————————— #
    write_model_stats_to_file(
        model=model,
        file_path=os.path.join(args_dir, "model_stats.txt"),
        batch_size=BATCH_SIZE,
        bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
        device="gpu/0",
        stats_to_measure=(
            "parameters",
            "model_size",
            "flops",
            "macs",
            "summary",
            "inference_latency",
            # "cpu_util_percent",
            # "cpu_power_rapl_w",
            # "ram_used_bytes",
            # "ram_util_percent",
            # "gpu_util_percent",
            # "gpu_mem_used_bytes",
            # "gpu_power_w",
        ),
        extra_attrs={
            "final_loss": history.history["loss"][-1],
            "final_val_loss": history.history["val_loss"][-1],
            "final_accuracy": history.history["accuracy"][-1],
            "final_val_accuracy": history.history["val_accuracy"][-1],
            "best_epoch": best_idx + 1,
            "best_train_loss": best_train_loss,
            "best_val_loss": best_val_loss,
            "s008_loss": float(s008_loss),
            "s009_loss": float(s009_loss),
            "best_train_accuracy": best_train_acc,
            "best_val_accuracy": best_val_acc,
            "s008_accuracy": float(s008_acc),
            "s009_accuracy": float(s009_acc),
        },
        test_runs=10,
        verbose=1,
    )
    # ———————————————————————————————————————————————————————————————————————————— #
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)
